# Brute Force Approach for Minimum fuel Trajectories in Earth Moon System

This notebook applies a brute force approach to solve the problem of launching a rocket from Low Earth Orbit (LEO) to Low Moon Orbit (LMO).

Two impulsive burns are appied. One at LEO and one at LMO.

The time of flight and phase of departure are also optimized

### Imports

In [7]:
import numpy as np
import pandas as pd
from pathlib import Path
from cr3bp import (
    create_earth_moon_system,
    grid_search_method,
    shrink_ranges,
    file_path
)

### Initialize the System / Problem

In [8]:
# Create the Earth-Moon system using the cr3bp module
em = create_earth_moon_system()
print(em.info())

CR3BP System Information:
  Primary 1 mass: 5.972e+24 kg
  Primary 2 mass: 7.342e+22 kg
  Primary 1 radius: 6.371e+06 m
  Primary 2 radius: 1.737e+06 m
  Total mass: 6.045e+24 kg
  Distance: 3.844e+08 m (384400.0 km)
  Mass parameter μ: 0.012145

Characteristic scales:
  Length (l*): 3.844e+08 m (384400.0 km)
  Time (t*): 3.752e+05 s (4.343 days)
  Velocity (v*): 1.025e+03 m/s (1.025 km/s)
  Acceleration (a*): 2.731e-03 m/s^2
  Period: 27.285 days
None


In [9]:
# Define the LEO and LMO altitudes in meters
leo_alt_m=463e3
lmo_alt_m=100e3

In [10]:
# 10km in natural units
print(f"10km in natural units = {1e3/em.l_star}")

10km in natural units = 2.6014568158168575e-06


## Run the Optimization Method

In [11]:
grid_size = (20, 20, 20, 20)  # (num_theta, num_delta_v, num_delta_v_angle, num_tof)

### Iteration 1 - Coarse Grid Search

In [12]:
dec_var_ranges = [[3.9, 4.0], [3.0, 3.1], [0.08, 0.15], [0.68, 0.80]]

In [13]:
optimals_iteration1 = Path(f"{file_path}/optimals_iteration1.npy")

if optimals_iteration1.exists():
    optimals_iteration1 = np.load(optimals_iteration1, allow_pickle=True)
    print("Loaded existing results")
    print(optimals_iteration1)
else:
    results_df = grid_search_method(em, grid_size, dec_var_ranges, 2.6e-5, leo_alt_m, lmo_alt_m)
    optimals_iteration1 = results_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
    np.save(f"{file_path}/optimals_iteration1.npy", optimals_iteration1)
    np.save(f"{file_path}/optimals_iteration1_df.npy", results_df)   
    print("Ran grid search and saved")
    print(optimals_iteration1)
    print(results_df)

Loaded existing results
[3.98947368 3.02105263 0.08736842 0.77473684]


### Iteration 2

In [14]:
optimals_iteration2 = Path(f"{file_path}/optimals_iteration2.npy")

if optimals_iteration2.exists():
    dec_var_ranges2 = shrink_ranges(optimals_iteration1, dec_var_ranges, shrink_factor=0.5)
    optimals_iteration2_df = np.load(f"{file_path}/optimals_iteration2_df.npy", allow_pickle=True)
    print(f"New dec_var_ranges2: {dec_var_ranges2}")
    optimals_iteration2 = np.load(optimals_iteration2, allow_pickle=True)
    print("Loaded existing results")
    print(optimals_iteration2)
else:
    dec_var_ranges2 = shrink_ranges(optimals_iteration1, dec_var_ranges, shrink_factor=0.5)
    print(f"New dec_var_ranges2: {dec_var_ranges2}")
    optimals_iteration2_df = grid_search_method(em, grid_size, dec_var_ranges2, 9e-6, leo_alt_m, lmo_alt_m)
    optimals_iteration2 = optimals_iteration2_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
    np.save(f"{file_path}/optimals_iteration2.npy", optimals_iteration2)
    np.save(f"{file_path}/optimals_iteration2_df.npy", optimals_iteration2_df)   
    print("Ran grid search and saved")
    print(optimals_iteration2)
    print(optimals_iteration2_df)

New dec_var_ranges2: [[np.float64(3.9644736842105264), 4.0], [3.0, np.float64(3.0460526315789473)], [0.08, np.float64(0.10486842105263158)], [np.float64(0.7447368421052631), 0.8]]
Loaded existing results
[3.97943213 3.02666205 0.10486842 0.76800554]


### Iteration 3

In [15]:
optimals_iteration3 = Path(f"{file_path}/optimals_iteration3.npy")

if optimals_iteration3.exists():
    optimals_iteration3 = np.load(optimals_iteration3, allow_pickle=True)
    optimals_iteration3_df = np.load(f"{file_path}/optimals_iteration3_df.npy", allow_pickle=True)
    print("Loaded existing results")
    print(optimals_iteration3)
else:
    dec_var_ranges3 = shrink_ranges(optimals_iteration2, dec_var_ranges2, shrink_factor=0.5)
    print(f"New dec_var_ranges3: {dec_var_ranges3}")
    print(f"Optimal iteration 2: {optimals_iteration2}")
    optimals_iteration3_df = grid_search_method(em, grid_size, dec_var_ranges3, 3e-6, leo_alt_m, lmo_alt_m)
    optimals_iteration3 = optimals_iteration3_df.iloc[0][['theta', 'delta_v', 'delta_v_angle', 'tof']].values
    np.save(f"{file_path}/optimals_iteration3.npy", optimals_iteration3)
    np.save(f"{file_path}/optimals_iteration3_df.npy", optimals_iteration3_df)   
    print("Ran grid search and saved")
    print(optimals_iteration3)
    print(optimals_iteration3_df)

Loaded existing results
[3.9836392  3.02605609 0.1045412  0.77454986]


In [17]:
dec_var_ranges2 = shrink_ranges(optimals_iteration1, dec_var_ranges, shrink_factor=0.5)
dec_var_ranges3 = shrink_ranges(optimals_iteration2, dec_var_ranges2, shrink_factor=0.5)

In [ ]:
for dec_range in dec_var_ranges3:
    differences = np.diff(dec_range)
print(differences)

[0.02763158]
